In [ ]:
# Enable the R environment in Google Colab
%load_ext rpy2.ipython

In [ ]:
%%R
# Install the necessary statistical packages
install.packages(c("afex", "emmeans", "dplyr", "rstatix"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘colorspace’, ‘fracdiff’, ‘lmtest’, ‘timeDate’, ‘urca’, ‘zoo’, ‘RcppArmadillo’, ‘rbibutils’, ‘cowplot’, ‘Deriv’, ‘forecast’, ‘SparseM’, ‘MatrixModels’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘RcppEigen’, ‘doBy’, ‘carData’, ‘abind’, ‘Formula’, ‘quantreg’, ‘plyr’, ‘lme4’, ‘pbkrtest’, ‘lmerTest’, ‘car’, ‘reshape2’, ‘reformulas’, ‘estimability’, ‘mvtnorm’, ‘numDeriv’, ‘corrplot’

trying URL 'https://cran.rstudio.com/src/contrib/colorspace_2.1-3.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/fracdiff_1.5-4.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/lmtest_0.9-40.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/timeDate_4052.112.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/urca_1.3-4.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/zoo_1.9-0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RcppArmadillo_15.4.2-1.tar.gz'
trying U

In [ ]:
%%R
# Load required libraries
library(dplyr)
library(afex)
library(emmeans)

# 1. Load the 384-row SA dataset
raw_sa <- read.csv("data_for_driver_SA.csv")

# 2. We first filter by Level == 1 to drop the sub-level rows per trial for the overall SA analysis
sa_dat <- raw_sa %>% filter(Level == 1)

# 3. Calculate the SA_change metric
sa_dat$SA_change <- sa_dat$Overall_SA - sa_dat$Initial_SA

# 4. Convert categorical variables to factors (CRUCIAL for ANOVA)
sa_dat$Participant <- as.factor(sa_dat$Participant)
sa_dat$Group <- as.factor(sa_dat$Group)
sa_dat$Attack <- as.factor(sa_dat$Attack)
sa_dat$Reliability <- as.factor(sa_dat$Reliability)
sa_dat$Order <- as.factor(sa_dat$Order)

cat("Data transformed successfully!\n")
cat("Total rows (should be exactly 128):", nrow(sa_dat), "\n")

Data transformed successfully!
Total rows (should be exactly 128): 128 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Loading required package: lme4
Loading required package: Matrix
Registered S3 method overwritten by 'car':
  method           from
  na.action.merMod lme4
************
Welcome to afex. For support visit: http://afex.singmann.science/
- Functions for ANOVAs: aov_car(), aov_ez(), and aov_4()
- Methods for calculating p-values with mixed(): 'S', 'KR', 'LRT', and 'PB'
- 'afex_aov' and 'mixed' objects can be passed to emmeans() for follow-up tests
- Get and set global package options with: afex_options()
- Set sum-to-zero contrasts globally: set_sum_contrasts()
- For example analyses see: browseVignettes("afex")
************

Attaching package: ‘afex’

The following object is masked from ‘package:lme4’:

    lmer

Welcome to emmeans.
Caution: You lose important information if you filter this pack

In [ ]:
%%R
# Load required libraries
library(dplyr)

cat("--- SECTION 4.1: DRIVER SA (OVERALL) DESCRIPTIVE STATISTICS ---\n\n")

# ---------------------------------------------------------
# 0. Prepare Baseline data
# ---------------------------------------------------------
baseline_dat <- sa_dat %>%
  group_by(Participant, Group) %>%
  summarise(Initial_SA = first(Initial_SA), .groups = 'drop')

# ---------------------------------------------------------
# 1. Baseline SA by Group
# ---------------------------------------------------------
cat("[1] Baseline Initial SA by Group (0 = No Explanation, 1 = With Explanation):\n")
desc_baseline <- baseline_dat %>%
  group_by(Group) %>%
  summarise(
    N_Participants = n(),
    Mean_Initial_SA = round(mean(Initial_SA), 2),
    SD_Initial_SA = round(sd(Initial_SA), 2),
    SE_Initial_SA = round(sd(Initial_SA) / sqrt(n()), 3)
  )
print(as.data.frame(desc_baseline))

# ---------------------------------------------------------
# 2. SA Change by Attack Object
# ---------------------------------------------------------
cat("\n[2] SA Change by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):\n")
desc_attack <- sa_dat %>%
  group_by(Attack) %>%
  summarise(
    N_Trials = n(),
    Mean_SA_Change = round(mean(SA_change), 2),
    SD_SA_Change = round(sd(SA_change), 2),
    SE_SA_Change = round(sd(SA_change) / sqrt(n()), 3)
  )
print(as.data.frame(desc_attack))

# ---------------------------------------------------------
# 3. SA Change by Group AND Attack Object
# ---------------------------------------------------------
cat("\n[3] SA Change by Group and Attack Object:\n")
desc_group_attack <- sa_dat %>%
  group_by(Group, Attack) %>%
  summarise(
    N_Trials = n(),
    Mean_SA_Change = round(mean(SA_change), 2),
    SD_SA_Change = round(sd(SA_change), 2),
    SE_SA_Change = round(sd(SA_change) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_group_attack))

# ---------------------------------------------------------
# 4. SA Change by System Reliability
# ---------------------------------------------------------
cat("\n[4] SA Change by System Reliability (Groups Merged, Participant-Averaged):\n")
desc_rel_merged <- sa_dat %>%
  # STEP 1: Average the 3 attacks together for Rel=0 for each participant
  group_by(Participant, Reliability) %>%
  summarise(Participant_Avg_SA = mean(SA_change), .groups = 'drop') %>%

  # STEP 2: Calculate the final descriptive stats
  group_by(Reliability) %>%
  summarise(
    N_Participants = n(),  # This will now correctly say 32!
    Mean_SA_Change = round(mean(Participant_Avg_SA), 2),
    SD_SA_Change = round(sd(Participant_Avg_SA), 2),
    SE_SA_Change = round(sd(Participant_Avg_SA) / sqrt(n()), 3)
  )
print(as.data.frame(desc_rel_merged))

# ---------------------------------------------------------
# 5. SA Change by Group AND System Reliability
# ---------------------------------------------------------
cat("\n[5] SA Change by Group AND System Reliability (Participant-Averaged):\n")
desc_group_rel <- sa_dat %>%
  # STEP 1: Average the 3 attacks together for Rel=0 for each participant, keeping Group
  group_by(Participant, Group, Reliability) %>%
  summarise(Participant_Avg_SA = mean(SA_change), .groups = 'drop') %>%

  # STEP 2: Calculate descriptive stats grouped by BOTH Group and Reliability
  group_by(Group, Reliability) %>%
  summarise(
    N_Participants = n(),
    Mean_SA_Change = round(mean(Participant_Avg_SA), 2),
    SD_SA_Change = round(sd(Participant_Avg_SA), 2),
    SE_SA_Change = round(sd(Participant_Avg_SA) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_group_rel))

# ---------------------------------------------------------
# 6. SA Change by Trial Order
# ---------------------------------------------------------
cat("\n[6] SA Change by Trial Order (Trials 1 through 4):\n")
desc_order <- sa_dat %>%
  group_by(Order) %>%
  summarise(
    N_Trials = n(),
    Mean_SA_Change = round(mean(SA_change), 2),
    SD_SA_Change = round(sd(SA_change), 2),
    SE_SA_Change = round(sd(SA_change) / sqrt(n()), 3)
  )
print(as.data.frame(desc_order))

--- SECTION 4.1: DRIVER SA (OVERALL) DESCRIPTIVE STATISTICS ---

[1] Baseline Initial SA by Group (0 = No Explanation, 1 = With Explanation):
  Group N_Participants Mean_Initial_SA SD_Initial_SA SE_Initial_SA
1     0             16            2.81          0.75         0.188
2     1             16            2.25          1.06         0.266

[2] SA Change by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):
  Attack N_Trials Mean_SA_Change SD_SA_Change SE_SA_Change
1      1       32           0.19         0.93        0.165
2      2       32           0.19         0.93        0.165
3      3       32           0.16         1.11        0.196
4      4       32          -0.19         1.35        0.239

[3] SA Change by Group and Attack Object:
  Group Attack N_Trials Mean_SA_Change SD_SA_Change SE_SA_Change
1     0      1       16          -0.12         0.50        0.125
2     0      2       16          -0.12         0.50        0.125
3     0      3       16          -0

In [ ]:
%%R

cat("--- SECTION 4.1: DRIVER SA (OVERALL) INFERENTIAL STATISTIC---\n\n")

# ---------------------------------------------------------
# 1. Baseline SA Comparison (Student's t-test)
# ---------------------------------------------------------

baseline_dat <- sa_dat[!duplicated(sa_dat$Participant), ]

# Run Student's t-test (var.equal = TRUE, prevents Welch's adjustment)
baseline_test <- t.test(Initial_SA ~ Group, data = baseline_dat, var.equal = TRUE)
cat("\n[TEST 1] Baseline SA Comparison:\n")
print(baseline_test)

# ---------------------------------------------------------
# 2. Effects of Explanation and Attack Object
# ---------------------------------------------------------
aov_sa_attack <- aov_ez(
  id = "Participant",
  dv = "SA_change",
  between = "Group",
  within = "Attack",
  data = sa_dat
)
cat("\n[TEST 2] SA_change by Explanation and Attack Object:\n")
print(nice(aov_sa_attack, es = "pes"))

# ---------------------------------------------------------
# 3. Effects of System Reliability
# ---------------------------------------------------------
aov_sa_rel <- aov_ez(
  id = "Participant",
  dv = "SA_change",
  between = "Group",
  within = "Reliability",
  data = sa_dat
)
cat("\n[TEST 3] SA_change by Explanation and Reliability:\n")
print(nice(aov_sa_rel,  es = "pes"))

# ---------------------------------------------------------
# 4. Exploratory Analysis: Trial Order (1 to 4)
# ---------------------------------------------------------
aov_sa_order <- aov_ez(
  id = "Participant",
  dv = "SA_change",
  within = "Order",
  data = sa_dat
)
cat("\n[TEST 4] SA_change by Trial Order (1 to 4):\n")
print(nice(aov_sa_order,  es = "pes"))

--- SECTION 4.1: DRIVER SA (OVERALL) INFERENTIAL STATISTIC---


[TEST 1] Baseline SA Comparison:

	Two Sample t-test

data:  Initial_SA by Group
t = 1.7278, df = 30, p-value = 0.09431
alternative hypothesis: true difference in means between group 0 and group 1 is not equal to 0
95 percent confidence interval:
 -0.1023831  1.2273831
sample estimates:
mean in group 0 mean in group 1 
         2.8125          2.2500 


[TEST 2] SA_change by Explanation and Attack Object:
Anova Table (Type 3 tests)

Response: SA_change
        Effect          df  MSE      F  pes p.value
1        Group       1, 30 3.18 4.97 * .142    .033
2       Attack 2.36, 70.73 0.52 2.63 + .081    .070
3 Group:Attack 2.36, 70.73 0.52   0.79 .026    .477
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 

[TEST 3] SA_change by Explanation and Reliability:
Anova Table (Type 3 tests)

Response: SA_change
             Effect    df  MSE      F   pes p.value
1             Gro

Contrasts set to contr.sum for the following variables: Group
Contrasts set to contr.sum for the following variables: Group
In addition: Warning message:
More than one observation per design cell, aggregating data using `fun_aggregate = mean`.
To turn off this warning, pass `fun_aggregate = mean` explicitly. 


In [ ]:
%%R
# Install rstatix quietly if missing

library(rstatix)

# Suppress summarize grouping warnings
options(dplyr.summarise.inform = FALSE)

cat("--- SECTION 4.1: DRIVER SA (3-LEVELS) ANALYSIS ---\n\n")

# ---------------------------------------------------------
# 0: Prepare 3-level data
# ---------------------------------------------------------

# 1. Load the full 384-row trial dataset
trial_data <- read.csv("data_for_driver_SA.csv")

# 2. Calculate the each level SA change score
trial_data$Level_SA_change <- trial_data$Level_SA - trial_data$Initial_level

# 3. Calculate mean change score and mutate all required columns
agg_data <- trial_data %>%
  group_by(Participant, Group, Level) %>%
  summarise(Mean_SA_Change = mean(Level_SA_change), .groups = "drop") %>%
  mutate(
    Level = as.factor(Level),
    Participant = as.factor(Participant),
    Outcome = ifelse(Mean_SA_Change < 0, "Drop", "Stable")
  )


# ---------------------------------------------------------
# 1: DESCRIPTIVE STATISTICS
# ---------------------------------------------------------
cat("[1] Descriptive Participant-Level Means (M) for SA Change:\n\n")

desc_table <- agg_data %>%
  group_by(Group, Level) %>%
  summarise(
    N_Participants = n(),
    M = round(mean(Mean_SA_Change), 3),
    SD = round(sd(Mean_SA_Change), 3),
    .groups = "drop"
  )

print(as.data.frame(desc_table))


# ---------------------------------------------------------
# 2: STATISTICAL TESTS BY GROUP
# ---------------------------------------------------------
for (g in c(0, 1)) {
  cat("\n[2] Group-Level Analysis for SA Change:\n\n")

  # Filter data for current group
  group_df <- agg_data %>% filter(Group == g)

  cat("--- [A] Repeated Measures ANOVA ---\n")
  tryCatch({
    res.aov <- anova_test(data = group_df, dv = Mean_SA_Change, wid = Participant, within = Level)
    print(as.data.frame(get_anova_table(res.aov)))
  }, error = function(e) cat("Error running ANOVA:", conditionMessage(e), "\n"))


  cat("\n--- [B] Chi-Square Test ---\n")
  tryCatch({
    contingency_table <- table(Level = group_df$Level, Outcome = group_df$Outcome)
    cat("\n    >> Observed Frequencies (Count):\n")
    print(contingency_table)

    chi_res <- chisq.test(contingency_table, simulate.p.value = TRUE)
    cat(sprintf("\n    >> Chi-Square Stats:\nChi2: %.4f, simulated p-value: %.4f\n",
                chi_res$statistic, chi_res$p.value))
  }, error = function(e) cat("Error running Chi-Square:", conditionMessage(e), "\n"))
}

--- SECTION 4.1: DRIVER SA (3-LEVELS) ANALYSIS ---

[1] Descriptive Participant-Level Means (M) for SA Change:

  Group Level N_Participants      M    SD
1     0     1             16 -0.031 0.202
2     0     2             16 -0.109 0.182
3     0     3             16 -0.125 0.158
4     1     1             16  0.078 0.373
5     1     2             16  0.203 0.458
6     1     3             16  0.156 0.482

[2] Group-Level Analysis for SA Change:

--- [A] Repeated Measures ANOVA ---
  Effect DFn DFd     F     p p<.05   ges
1  Level   2  30 2.888 0.071       0.052

--- [B] Chi-Square Test ---

    >> Observed Frequencies (Count):
     Outcome
Level Drop Stable
    1    3     13
    2    5     11
    3    7      9

    >> Chi-Square Stats:
Chi2: 2.3273, simulated p-value: 0.3853

[2] Group-Level Analysis for SA Change:

--- [A] Repeated Measures ANOVA ---
  Effect DFn DFd     F     p p<.05   ges
1  Level   2  30 1.046 0.364       0.014

--- [B] Chi-Square Test ---

    >> Observed Frequencie


Attaching package: ‘rstatix’

The following object is masked from ‘package:stats’:

    filter



In [ ]:
%%R

# Load required libraries
library(dplyr)
library(afex)

cat("--- SECTION 4.2: TAKEOVER PERFORMANCE DESCRIPTIVE STATISTIC ---\n\n")

# 1. Load the Takeover Performance dataset
perf_data <- read.csv("data_for_takeover_performance.csv")

# 2. Filter to Level == 1 to get exactly 128 unique driving trials
# (Since RT is measured once per attack, not per SA sub-level)
perf_trials <- perf_data %>% filter(Level == 1)

# 3. Convert categorical variables to factors
perf_trials$Participant <- as.factor(perf_trials$Participant)
perf_trials$Group <- as.factor(perf_trials$Group)
perf_trials$Attack <- as.factor(perf_trials$Attack)
perf_trials$Reliability <- as.factor(perf_trials$Reliability)
perf_trials$Order <- as.factor(perf_trials$Order)


# ---------------------------------------------------------
# [1] RT by Attack Object
# ---------------------------------------------------------
cat("\n[1] RT (Longitudinal & Lateral) by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):\n")
desc_attack <- perf_trials %>%
  group_by(Attack) %>%
  summarise(
    N_Trials = n(),
    Mean_RT_Long = round(mean(RT_Longitudinal), 2),
    SE_RT_Long = round(sd(RT_Longitudinal) / sqrt(n()), 3),
    Mean_RT_Lat = round(mean(RT_Lateral), 2),
    SE_RT_Lat = round(sd(RT_Lateral) / sqrt(n()), 3)
  )
print(as.data.frame(desc_attack))

# ---------------------------------------------------------
# [2] RT by Group and Attack Object
# ---------------------------------------------------------
cat("\n[2] RT (Longitudinal & Lateral) by Group and Attack Object:\n")
desc_group_attack <- perf_trials %>%
  group_by(Group, Attack) %>%
  summarise(
    N_Trials = n(),
    Mean_RT_Long = round(mean(RT_Longitudinal), 2),
    SE_RT_Long = round(sd(RT_Longitudinal) / sqrt(n()), 3),
    Mean_RT_Lat = round(mean(RT_Lateral), 2),
    SE_RT_Lat = round(sd(RT_Lateral) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_group_attack))

# =========================================================
# PREPARE PARTICIPANT-AVERAGED DATA FOR RELIABILITY
# (Averages the 3 attacks for Rel=0 for each participant)
# =========================================================
participant_rt_rel <- perf_trials %>%
  group_by(Participant, Group, Reliability) %>%
  summarise(
    Avg_RT_Long = mean(RT_Longitudinal),
    Avg_RT_Lat = mean(RT_Lateral),
    .groups = 'drop'
  )

# ---------------------------------------------------------
# [3] RT_Longitudinal by System Reliability
# ---------------------------------------------------------
cat("\n[3] RT_Longitudinal by System Reliability (Groups Merged, Participant-Averaged):\n")
desc_rel_long <- participant_rt_rel %>%
  group_by(Reliability) %>%
  summarise(
    N_Participants = n(), # This will now equal 32
    Mean_RT_Long = round(mean(Avg_RT_Long), 2),
    SE_RT_Long = round(sd(Avg_RT_Long) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_rel_long))

# ---------------------------------------------------------
# [4] RT_Lateral by System Reliability
# ---------------------------------------------------------
cat("\n[4] RT_Lateral by System Reliability (Groups Merged, Participant-Averaged):\n")
desc_rel_lat <- participant_rt_rel %>%
  group_by(Reliability) %>%
  summarise(
    N_Participants = n(),
    Mean_RT_Lat = round(mean(Avg_RT_Lat), 2),
    SE_RT_Lat = round(sd(Avg_RT_Lat) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_rel_lat))

# ---------------------------------------------------------
# [5] RT by Group AND Reliability
# ---------------------------------------------------------
cat("\n[5] RT (Longitudinal & Lateral) by Group and Reliability (Participant-Averaged):\n")
desc_group_rel <- participant_rt_rel %>%
  group_by(Group, Reliability) %>%
  summarise(
    N_Participants = n(),
    Mean_RT_Long = round(mean(Avg_RT_Long), 2),
    SE_RT_Long = round(sd(Avg_RT_Long) / sqrt(n()), 3),
    Mean_RT_Lat = round(mean(Avg_RT_Lat), 2),
    SE_RT_Lat = round(sd(Avg_RT_Lat) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_group_rel))

--- SECTION 4.2: TAKEOVER PERFORMANCE DESCRIPTIVE STATISTIC ---


[1] RT (Longitudinal & Lateral) by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):
  Attack N_Trials Mean_RT_Long SE_RT_Long Mean_RT_Lat SE_RT_Lat
1      1       32         2.88      0.175        5.10     0.481
2      2       32         2.65      0.236        5.53     0.588
3      3       32         2.64      0.124        6.52     0.568
4      4       32         3.26      0.272        5.85     0.385

[2] RT (Longitudinal & Lateral) by Group and Attack Object:
  Group Attack N_Trials Mean_RT_Long SE_RT_Long Mean_RT_Lat SE_RT_Lat
1     0      1       16         2.60      0.226        4.54     0.447
2     0      2       16         2.38      0.271        5.92     1.049
3     0      3       16         2.52      0.182        6.64     0.917
4     0      4       16         2.93      0.367        5.72     0.654
5     1      1       16         3.17      0.255        5.66     0.846
6     1      2       16    

In [ ]:
%%R
# Load required libraries
library(dplyr)
library(afex)

cat("--- SECTION 4.2: TAKEOVER PERFORMANCE DESCRIPTIVE STATISTIC ---\n\n")

# 1. Load the Takeover Performance dataset
perf_data <- read.csv("data_for_takeover_performance.csv")

# 2. Filter to Level == 1 to get exactly 128 unique driving trials
# (Since RT is measured once per attack, not per SA sub-level)
perf_trials <- perf_data %>% filter(Level == 1)

# 3. Convert categorical variables to factors
perf_trials$Participant <- as.factor(perf_trials$Participant)
perf_trials$Group <- as.factor(perf_trials$Group)
perf_trials$Attack <- as.factor(perf_trials$Attack)
perf_trials$Reliability <- as.factor(perf_trials$Reliability)
perf_trials$Order <- as.factor(perf_trials$Order)



cat("\n[1] RT (longitudinal&Lateral) by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):\n")
desc_attack <- perf_trials %>%
  group_by(Attack) %>%
  summarise(
    N_Trials = n(),
    Mean_RT_Long = round(mean(RT_Longitudinal), 2),
    SE_RT_Long = round(sd(RT_Longitudinal) / sqrt(n()), 3),
    Mean_RT_Lat = round(mean(RT_Lateral), 2),
    SE_RT_Lat = round(sd(RT_Lateral) / sqrt(n()), 3)
  )
print(as.data.frame(desc_attack))

cat("\n[2] RT_Longitudinal by System Reliability (Groups Merged):\n")
desc_rel <- perf_trials %>%
  group_by(Reliability) %>%
  summarise(
    Mean_RT_Long = round(mean(RT_Longitudinal), 2),
    SE_RT_Long = round(sd(RT_Longitudinal) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_rel))

cat("\n[3] RT_Lateral by System Reliability (Groups Merged):\n")
desc_rel <- perf_trials %>%
  group_by(Reliability) %>%
  summarise(
    Mean_RT_Lat = round(mean(RT_Lateral), 2),
    SE_RT_Lat = round(sd(RT_Lateral) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_rel))

--- SECTION 4.2: TAKEOVER PERFORMANCE DESCRIPTIVE STATISTIC ---


[1] RT (longitudinal&Lateral) by Attack Object (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings):
  Attack N_Trials Mean_RT_Long SE_RT_Long Mean_RT_Lat SE_RT_Lat
1      1       32         2.88      0.175        5.10     0.481
2      2       32         2.65      0.236        5.53     0.588
3      3       32         2.64      0.124        6.52     0.568
4      4       32         3.26      0.272        5.85     0.385

[2] RT_Longitudinal by System Reliability (Groups Merged):
  Reliability Mean_RT_Long SE_RT_Long
1           0         2.82      0.126
2           1         2.97      0.197

[3] RT_Lateral by System Reliability (Groups Merged):
  Reliability Mean_RT_Lat SE_RT_Lat
1           0        5.78     0.292
2           1        5.68     0.545


In [ ]:
%%R
cat("--- SECTION 4.2: TAKEOVER PERFORMANCE (ANOVAs) ---\n\n")

# =========================================================
# LONGITUDINAL REACTION TIME
# =========================================================
cat("=========== DV: LONGITUDINAL RT ===========\n")

# 1. Effects of Explanation and Attack Object
aov_long_attack <- aov_ez(
  id = "Participant",
  dv = "RT_Longitudinal",
  between = "Group",
  within = "Attack",
  data = perf_trials
)
cat("\n[TEST 1] RT_Longitudinal by Explanation and Attack Object\n")
print(nice(aov_long_attack, es = "pes"))

# 2. Effects of Explanation and Reliability
aov_long_rel <- aov_ez(
  id = "Participant",
  dv = "RT_Longitudinal",
  between = "Group",
  within = "Reliability",
  data = perf_trials
)
cat("\n[TEST 2] RT_Longitudinal by Explanation and Reliability\n")
print(nice(aov_long_rel, es = "pes"))


# =========================================================
# LATERAL REACTION TIME
# =========================================================
cat("\n\n=========== DV: LATERAL RT ===========\n")

# 1. Effects of Explanation and Attack Object
aov_lat_attack <- aov_ez(
  id = "Participant",
  dv = "RT_Lateral",
  between = "Group",
  within = "Attack",
  data = perf_trials
)
cat("\n[TEST 3] RT_Lateral by Explanation and Attack\n")
print(nice(aov_lat_attack, es = "pes"))

# 2. Effects of Explanation and Reliability
aov_lat_rel <- aov_ez(
  id = "Participant",
  dv = "RT_Lateral",
  between = "Group",
  within = "Reliability",
  data = perf_trials
)
cat("\n[TEST 4] RT_Lateral by Explanation and Reliability\n")
print(nice(aov_lat_rel, es = "pes"))

--- SECTION 4.2: TAKEOVER PERFORMANCE (ANOVAs) ---

=========== DV: LONGITUDINAL RT ===========

[TEST 1] RT_Longitudinal by Explanation and Attack Object
Anova Table (Type 3 tests)

Response: RT_Longitudinal
        Effect          df  MSE      F  pes p.value
1        Group       1, 30 3.50   2.32 .072    .138
2       Attack 2.44, 73.34 0.82 4.10 * .120    .014
3 Group:Attack 2.44, 73.34 0.82   0.42 .014    .701
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 

[TEST 2] RT_Longitudinal by Explanation and Reliability
Anova Table (Type 3 tests)

Response: RT_Longitudinal
             Effect    df  MSE    F  pes p.value
1             Group 1, 30 1.80 2.65 .081    .114
2       Reliability 1, 30 0.29 1.12 .036    .299
3 Group:Reliability 1, 30 0.29 0.40 .013    .534
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1


=========== DV: LATERAL RT ===========

[TEST 3] RT_Lateral by Explanation and Attack
Anova Table (Type 3

Contrasts set to contr.sum for the following variables: Group
Contrasts set to contr.sum for the following variables: Group
Contrasts set to contr.sum for the following variables: Group
Contrasts set to contr.sum for the following variables: Group
In addition: Warning messages:
1: More than one observation per design cell, aggregating data using `fun_aggregate = mean`.
To turn off this warning, pass `fun_aggregate = mean` explicitly. 
2: More than one observation per design cell, aggregating data using `fun_aggregate = mean`.
To turn off this warning, pass `fun_aggregate = mean` explicitly. 


In [ ]:
%%R
# Load required libraries
library(emmeans)

cat("--- SECTION 4.2: TAKEOVER PERFORMANCE (POST-HOC TESTS) ---\n\n")


# =========================================================
# PART A: LONGITUDINAL RT - RELIABILITY (Groups Merged)
# =========================================================
cat("================ 1. LONGITUDINAL RT: SYSTEM RELIABILITY ================\n")

emms_long_rel <- emmeans(aov_long_rel, ~ Reliability)
cat("\n[Reliability: Mean and SE]\n")
print(emms_long_rel)

cat("\n[Reliability: Bonferroni Pairwise Comparison]\n")
print(pairs(emms_long_rel, adjust = "bonferroni"))


# =========================================================
# PART B: LONGITUDINAL RT - ATTACK OBJECTS (Groups Merged)
# =========================================================
cat("================ 2. LONGITUDINAL RT: ATTACK OBJECT ================\n")

emms_long_attack <- emmeans(aov_long_attack, ~ Attack)
cat("\n[Attack (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings): Mean and SE]\n")
print(emms_long_attack)

cat("\n[Pairwise Comparisons (Bonferroni Adjusted)]\n")
print(pairs(emms_long_attack, adjust = "bonferroni"))


# =========================================================
# PART C: LATERAL RT - RELIABILITY WITHIN GROUPS (Simple Main Effects)
# =========================================================
cat("================ 3. LATERAL RT: SYSTEM RELIABILITY ================\n")

emms_lat_grouped <- emmeans(aov_lat_rel, ~ Reliability | Group)
cat("\n[Reliability: Mean and SE]\n")
print(emms_lat_grouped)

cat("\n[Pairwise Comparisons (Bonferroni Adjusted)]\n")
# Tests Low vs High for Group 0, and then Low vs High for Group 1
print(pairs(emms_lat_grouped, adjust = "bonferroni"))

--- SECTION 4.2: TAKEOVER PERFORMANCE (POST-HOC TESTS) ---

================ 1. LONGITUDINAL RT: SYSTEM RELIABILITY ================

[Reliability: Mean and SE]
 Reliability emmean    SE df lower.CL upper.CL
 X0            2.82 0.170 30     2.48     3.17
 X1            2.97 0.191 30     2.57     3.36

Results are averaged over the levels of: Group 
Confidence level used: 0.95 

[Reliability: Bonferroni Pairwise Comparison]
 contrast estimate    SE df t.ratio p.value
 X0 - X1    -0.143 0.135 30  -1.058  0.2986

Results are averaged over the levels of: Group 
================ 2. LONGITUDINAL RT: ATTACK OBJECT ================

[Attack (1=Stop Sign, 2=Vehicle, 3=Pedestrian, 4=Lane Markings): Mean and SE]
 Attack emmean    SE df lower.CL upper.CL
 X1       2.88 0.171 30     2.54     3.23
 X2       2.65 0.235 30     2.17     3.13
 X3       2.64 0.124 30     2.38     2.89
 X4       3.26 0.270 30     2.71     3.81

Results are averaged over the levels of: Group 
Confidence level used: 0.95 



Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'


In [ ]:
%%R

# Load required libraries
library(dplyr)
library(afex)
library(emmeans)

cat("--- SECTION 4.3: TRUST AND MENTAL WORKLOAD STATISTIC ---\n\n")

# Load and format the data
rel_data <- read.csv("data_for_trust&workload.csv", check.names = FALSE)
rel_data$Participant_ID <- as.factor(rel_data$Participant_ID)
rel_data$Group <- as.factor(rel_data$Group)
rel_data$Reliability <- as.factor(rel_data$Reliability)


# =========================================================
# 1. TRUST STATISTICS
# =========================================================
cat("================ TRUST ================\n")

# --- DESCRIPTIVE STATS ---
cat("\n[Trust by Group and Reliability (Descriptives)]\n")
desc_trust <- rel_data %>%
  group_by(Group, Reliability) %>%
  summarise(
    N = n(),
    Mean_Trust = round(mean(Trust), 2),
    SD_Trust = round(sd(Trust), 2),
    SE_Trust = round(sd(Trust) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_trust))

# --- INFERENTIAL STATS (ANOVA) ---
aov_trust <- aov_ez(id = "Participant_ID", dv = "Trust", between = "Group", within = "Reliability", data = rel_data)

cat("\n[Warning Conditions (Group): Mean and SE]\n")
print(emmeans(aov_trust, ~ Group))

cat("\n[Reliability Conditions (Reliability): Mean and SE]\n")
print(emmeans(aov_trust, ~ Reliability))

cat("\n[Interaction (Group * Reliability): Mean and SE]\n")
print(emmeans(aov_trust, ~ Group * Reliability)) # <-- Added this interaction!

cat("\n[Trust ANOVA: F-values and Partial Eta Squared (pes)]\n")
print(nice(aov_trust, es = "pes"))


# =========================================================
# 2. MENTAL WORKLOAD STATISTICS
# =========================================================
cat("\n\n================ MENTAL WORKLOAD ================\n")

# --- DESCRIPTIVE STATS ---
cat("\n[Mental Workload by Group and Reliability (Descriptives)]\n")
desc_workload <- rel_data %>%
  group_by(Group, Reliability) %>%
  summarise(
    N = n(),
    Mean_MW = round(mean(Mental_Workload), 2),
    SD_MW = round(sd(Mental_Workload), 2),
    SE_MW = round(sd(Mental_Workload) / sqrt(n()), 3),
    .groups = 'drop'
  )
print(as.data.frame(desc_workload))

# --- INFERENTIAL STATS (ANOVA) ---
aov_workload <- aov_ez(id = "Participant_ID", dv = "Mental_Workload", between = "Group", within = "Reliability", data = rel_data)

cat("\n[Warning Conditions (Group): Mean and SE]\n")
print(emmeans(aov_workload, ~ Group))

cat("\n[Reliability Conditions (Reliability): Mean and SE]\n")
print(emmeans(aov_workload, ~ Reliability))

cat("\n[Interaction (Group * Reliability): Mean and SE]\n")
print(emmeans(aov_workload, ~ Group * Reliability)) # <-- Added this interaction!

cat("\n[Mental Workload ANOVA: F-values and Partial Eta Squared (pes)]\n")
print(nice(aov_workload, es = "pes"))

--- SECTION 4.3: TRUST AND MENTAL WORKLOAD STATISTIC ---

================ TRUST ================

[Trust by Group and Reliability (Descriptives)]
  Group Reliability  N Mean_Trust SD_Trust SE_Trust
1     0           0 16       4.47     1.18    0.295
2     0           1 16       4.67     1.09    0.272
3     1           0 16       4.71     0.58    0.146
4     1           1 16       4.79     0.67    0.168

[Warning Conditions (Group): Mean and SE]
 Group emmean    SE df lower.CL upper.CL
 0       4.57 0.197 30     4.17     4.97
 1       4.75 0.197 30     4.35     5.15

Results are averaged over the levels of: Reliability 
Confidence level used: 0.95 

[Reliability Conditions (Reliability): Mean and SE]
 Reliability emmean    SE df lower.CL upper.CL
 X0            4.59 0.164 30     4.25     4.92
 X1            4.73 0.160 30     4.40     5.06

Results are averaged over the levels of: Group 
Confidence level used: 0.95 

[Interaction (Group * Reliability): Mean and SE]
 Group Reliability em

Contrasts set to contr.sum for the following variables: Group
Contrasts set to contr.sum for the following variables: Group
